# Mk33 memory study: GPU runs

Runs the two quick experiments from `docs/mk33/MEMORY.md` and packages the results:

1. **Kernel benchmark**: Triton vs PyTorch for KV-cache eviction (time and bandwidth). About 5 minutes.
2. **Perplexity vs budget**: how much each KV policy hurts the model as the cache shrinks. About 20 to 40 minutes.

At the end, a zip of the results downloads to your computer. Send it back to Claude.

**Before starting:** Runtime > Change runtime type > **GPU** (L4 or A100 if available, T4 works too). Then Runtime > Run all.

## 1. Check the GPU and pick a model

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU, then run again."
name = torch.cuda.get_device_name()
mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {name}, {mem_gb:.0f} GB")

# 7B needs about 16 GB plus cache and fast bfloat16 (L4, A100). On a T4 use 3B in float16.
if mem_gb >= 22 and torch.cuda.is_bf16_supported():
    MODEL, DTYPE = "Qwen/Qwen2.5-7B-Instruct", "bfloat16"
else:
    MODEL, DTYPE = "Qwen/Qwen2.5-3B-Instruct", "float16"
print("Model:", MODEL, DTYPE)

## 2. Install Mk33

In [ ]:
%cd /content
!rm -rf Mk33 && git clone -q https://github.com/dulguuntsatsralt-cell/Mk33
%cd /content/Mk33
!pip install -q -e '.[mk33-kv,dev]'
!git log --oneline -3

## 3. Run the tests on the GPU

On a GPU this also runs the Triton kernel tests for real (not in the interpreter). Everything should pass.

In [ ]:
!python -m pytest tests/mk33 -q 2>&1 | tail -5

## 4. Kernel benchmark: Triton vs PyTorch

In [ ]:
!mkdir -p results
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv > results/gpu.txt
!python scripts/mk33/bench_kv_kernels.py --dtype float16 --out results/kv_kernels.json

## 5. Check that tool calling works with the local model

Downloads the model (a few minutes the first time) and sends one agent-style request. It should end with "Tool call parsed".

In [ ]:
!python scripts/mk33/kv_quality.py --model {MODEL} --dtype {DTYPE} --smoke 2>&1 | tee results/smoke.txt | tail -25

## 6. Perplexity vs budget

Scores the last 1,024 tokens of a long document (the Mk33 paper plus scaffold code) given the 8,192 tokens before it, for each policy and cache budget. The "vs full" column is the cost of forgetting; random eviction is the floor a real policy has to beat.

H2O needs eager attention, which stores attention scores for every layer, so it runs separately with smaller chunks to fit in memory.

In [ ]:
!python scripts/mk33/kv_quality.py --model {MODEL} --dtype {DTYPE} --context 8192 --continuation 1024 \
    --policies full,random,streaming,keynorm --budgets 512,1024,2048,4096 \
    --out results/kv_quality.json 2>&1 | grep -v Warning | tee results/kv_quality.md

In [ ]:
!python scripts/mk33/kv_quality.py --model {MODEL} --dtype {DTYPE} --context 8192 --continuation 1024 \
    --policies full,h2o --budgets 512,1024,2048,4096 --chunk 128 \
    --out results/kv_quality_h2o.json 2>&1 | grep -v Warning | tee results/kv_quality_h2o.md

## 7. Download the results

Send `mk33_gpu_results.zip` back to Claude.

In [ ]:
!cd /content/Mk33 && zip -qr /content/mk33_gpu_results.zip results
!ls -la results
from google.colab import files
files.download("/content/mk33_gpu_results.zip")

## Optional: a first agent run (long)

The main experiment. This runs one environment, one repeat, three modes at an equal 8K memory budget, to time a run before committing to the full grid. Expect roughly 30 to 90 minutes depending on the GPU. Colab may disconnect on long runs; rerunning the same cell resumes and skips finished runs.

In [ ]:
!mk33 bench-prepare --out bench/envs
!mk33 bench-run --backend hf --model {MODEL} --dtype {DTYPE} --window 8000 \
    --modes rlc,kv:full,kv:keynorm --tasks knapsack --repeats 1 --max-rounds 8 --out results/agent_pilot
!cat results/agent_pilot/report.md